# FASE 2. LIMPIEZA DE LOS DATOS - TRAFICO (Hugging Face)

## 1. Carga de datos originales

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("TFM_Trafico_Valencia") \
    .getOrCreate()

# Ruta del archivo unificado (son los datos originales solo que el campo intensity tiene el mismo tipo de datos en todos los meses)
ruta_trafico = "gs://tfm_unir1/trafico/espires_VLC2_unificado"

# Cargar los datos
df_raw = spark.read.parquet(ruta_trafico)

print("Registros originales:", df_raw.count())
print("Columnas:", len(df_raw.columns))

26/10/24 14:17:33 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


Registros originales: 106467612
Columnas: 17


## 2. Selección de columnas de interés 

In [2]:
# Columnas necesarias para el análisis
columnas = ["detector_id", "snapshot_datetime", "intensity", "year", "month"]

# Crear un nuevo DataFrame con los cuatro contaminantes
df_trafico = df_raw.select (columnas)

# Comprobar los registros seleccionados
df_trafico.printSchema()

print("Total de registros seleccionados:", df_trafico.count())

root
 |-- detector_id: string (nullable = true)
 |-- snapshot_datetime: timestamp (nullable = true)
 |-- intensity: double (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)



Total de registros seleccionados: 106467612


## 3. Separación de los datos según su frecuencia temporal

En el caso de tráfico los datos vienen en la misma frecuencia (cada 15 minutos). La agrupacion de datos por hora se realizara en etapas posteriores

## 4. Limpieza de los datos 

### Tratamiento de valores no válidos

In [3]:
import pandas as pd
from pyspark.sql import Window

# Eliminar filas sin identificador de espira
df_trafico_clean = df_trafico.filter (F.col("detector_id")!= "")

print("Total de registros seleccionados:", df_trafico_clean.count())

Total de registros seleccionados: 102905412


In [4]:
# Eliminar filas que se correspondan a huecos y datos congelados (informados por Hugging Face)
# Para identificar los días sin datos o con datos no fiables se utiliza el archivo de calidad publicado por los autores del dataset
import pandas as pd

# Quedarse solo con los días "dynamic" del archivo de calidad
url = "https://huggingface.co/datasets/femartip/valencia-urban-mobility/resolve/main/data_quality/quality_flags.csv"
qf = pd.read_csv(url)
dias_dinamicos = qf[(qf["stream"] == "espires_VLC2") & (qf["regime"] == "dynamic")]["date"].tolist()
df_trafico_clean = df_trafico_clean.filter(F.date_format("snapshot_datetime", "yyyy-MM-dd").isin(dias_dinamicos))

print("Total de registros seleccionados:", df_trafico_clean.count())

Total de registros seleccionados: 65675063


In [6]:
# Eliminar filas sin intensidad válida (nulos y -1)
df_trafico_clean = df_trafico_clean.filter(F.col("intensity").isNotNull() & (F.col("intensity") != -1))

print("Total de registros seleccionados:", df_trafico_clean.count())

Total de registros seleccionados: 53651868


In [7]:
# Eliminar valores extremos (errores de medición)
df_trafico_clean = df_trafico_clean.filter(F.col("intensity") <= 12000)

print("Total de registros seleccionados:", df_trafico_clean.count())

Total de registros seleccionados: 53651553


In [10]:
# Unificamos la informacion para que nos queden un registro por hora y espira.  
# Esto soluciona los datos que aparecen multiplicados. Utilizamos la ultima captura

df_trafico_clean = (df_trafico_clean
                        .withColumn("hora", F.date_trunc("hour", F.col("snapshot_datetime")- F.expr("INTERVAL 15 MINUTES")))
                        .groupBy ("detector_id", "hora")
                        .agg (F.max_by("intensity","snapshot_datetime").alias("intensity"))
                   )

print("Total de registros seleccionados:", df_trafico_clean.count())

Total de registros seleccionados: 13302592


In [13]:
# Debido a las espiras muertas y a aquellas con más del 50% de datos en 0, generamos un grupo estable de analisis
resumen = df_trafico_clean.groupBy("detector_id").agg(
    F.max(F.year("hora") == 2024).alias("tiene_2024"),
    F.max(F.year("hora") == 2025).alias("tiene_2025"),
    F.avg((F.col("intensity") == 0).cast("int")).alias("pct_ceros"))

espiras_estables = resumen.filter(
    F.col("tiene_2024") & F.col("tiene_2025") & (F.col("pct_ceros")<= 0.5)
).select("detector_id")

df_trafico_clean = df_trafico_clean.join(espiras_estables, "detector_id")

print("Total de registros seleccionados:", df_trafico_clean.count())


Total de registros seleccionados: 11650206


In [16]:
# Agregamos año y mes para particionar ya que se perdieron en la agrupacion de espiras y horas
df_trafico_clean = (df_trafico_clean.withColumn("year", F.year("hora"))
                    .withColumn("month", F.month("hora")))

### Verificación de trafico tras la limpieza

In [18]:
df_trafico_clean.agg(
    F.count("*").alias("registros"),
    F.countDistinct("detector_id").alias("espiras"),
    F.min("hora").alias("primera_hora"),
    F.max("hora").alias("ultima_hora"),
    F.min("year").alias("primer_año"),
    F.max("year").alias("ultimo_año"),
    F.min("intensity").alias("minimo"),
    F.max("intensity").alias("maximo")
).show(truncate=False)

+---------+-------+-------------------+-------------------+----------+----------+------+-------+
|registros|espiras|primera_hora       |ultima_hora        |primer_año|ultimo_año|minimo|maximo |
+---------+-------+-------------------+-------------------+----------+----------+------+-------+
|11650206 |965    |2023-03-09 16:00:00|2025-09-23 23:00:00|2023      |2025      |0.0   |12000.0|
+---------+-------+-------------------+-------------------+----------+----------+------+-------+



In [19]:
df_trafico_clean.groupBy(F.hour("hora").alias("h")).count().orderBy("h").show(24)

+---+------+
|  h| count|
+---+------+
|  0|478776|
|  1|490919|
|  2|479005|
|  3|456285|
|  4|448685|
|  5|429357|
|  6|465730|
|  7|490921|
|  8|489849|
|  9|487388|
| 10|488737|
| 11|489053|
| 12|486763|
| 13|488488|
| 14|487161|
| 15|493661|
| 16|490487|
| 17|496305|
| 18|497725|
| 19|497408|
| 20|496544|
| 21|491538|
| 22|493540|
| 23|535881|
+---+------+



In [20]:
df_trafico_clean.show(5)

+-----------+-------------------+---------+----+-----+
|detector_id|               hora|intensity|year|month|
+-----------+-------------------+---------+----+-----+
|       3210|2023-04-06 11:00:00|    586.0|2023|    4|
|       1512|2023-04-06 21:00:00|     90.0|2023|    4|
|       1436|2023-04-05 14:00:00|   1500.0|2023|    4|
|       3210|2023-04-13 07:00:00|     84.0|2023|    4|
|       3210|2023-04-13 22:00:00|     42.0|2023|    4|
+-----------+-------------------+---------+----+-----+
only showing top 5 rows



## 5. Estandarización de variables 

### Estandarización de las variables temporales

In [24]:
from pyspark.sql import functions as F

# Añadir variables temporales a los datos horarios
df_trafico_clean = (df_trafico_clean
    .withColumnRenamed("hora","fecha_inicio")
    .withColumn("anio", F.year("fecha_inicio"))
    .withColumn("mes", F.month("fecha_inicio"))
    .withColumn("dia", F.dayofmonth("fecha_inicio"))
    .withColumn("hora", F.hour("fecha_inicio"))
    .withColumn("dia_semana", F.dayofweek("fecha_inicio"))
    .drop ("year","month")
)


# Comprobar resultado
df_trafico_clean.show(5, truncate=False)

+-----------+-------------------+---------+----+---+---+----+----------+
|detector_id|fecha_inicio       |intensity|anio|mes|dia|hora|dia_semana|
+-----------+-------------------+---------+----+---+---+----+----------+
|3606       |2024-09-17 01:00:00|210.0    |2024|9  |17 |1   |3         |
|829        |2024-09-17 10:00:00|410.0    |2024|9  |17 |10  |3         |
|829        |2024-09-17 22:00:00|158.0    |2024|9  |17 |22  |3         |
|1512       |2024-09-02 23:00:00|42.0     |2024|9  |2  |23  |2         |
|2904       |2024-09-24 23:00:00|0.0      |2024|9  |24 |23  |3         |
+-----------+-------------------+---------+----+---+---+----+----------+
only showing top 5 rows



### Verificación de la zona horaria

In [25]:
# Consultar la zona horaria utilizada por Spark
zona_horaria = spark.conf.get("spark.sql.session.timeZone")

print("Zona horaria de Spark:", zona_horaria)

Zona horaria de Spark: Etc/UTC


La sesión de Spark está configurada en UTC, pero los datos de tráfico conservan la hora local de València, tal como los publica la fuente. Antes de cruzarlos con otros datasets, se verificará que ambos usen la misma referencia horaria.

### Normalización de nombres de columnas

In [26]:
# Diccionario de nombres originales y estandarizados
columnas = {
    "detector_id": "id_espira",
    "intensity": "intensidad"
 }

# Función para renombrar columnas
def renombrar_columnas(df):
    for original, nuevo in columnas.items():
        df = df.withColumnRenamed(original, nuevo)
    return df

# Aplicar a ambos conjuntos
df_trafico_clean = renombrar_columnas(df_trafico_clean)

# Comprobar las columnas resultantes
print("Columnas del dataset trafico:")
print(df_trafico_clean.columns)

Columnas del dataset trafico:
['id_espira', 'fecha_inicio', 'intensidad', 'anio', 'mes', 'dia', 'hora', 'dia_semana']


## 7. Validación final de los datos preparados

In [30]:
# Comparar los registros antes y después de la limpieza
# En el caso de trafico tendremos diferencias porque hemos borrado y agrupado datos
datos_originales = df_trafico.count()
datos_limpios = df_trafico_clean.count()

print("Registros originales:", datos_originales)
print("Registros preparados:", datos_limpios)
print("Reducción:", datos_originales - datos_limpios,  f"({(1 - datos_limpios / datos_originales) * 100:.1f} %)")


Registros originales: 106467612
Registros preparados: 11650206
Reducción: 94817406 (89.1 %)


**Conclusiones:**
- La limpieza reduce el conjunto de 106.467.612 a 11.650.206 registros (−89,1 %). 
- La mayor parte de la reducción corresponde a capturas repetidas de una misma medición horaria; el resto, a registros sin intensidad válida, espiras sin datos o atascadas, días congelados y valores extremos

## 8. Exportación de los datasets limpios

In [31]:
# Rutas de salida
ruta_trafico = "gs://tfm_unir1/processed/trafico/espiras/"

# Guardar datos diarios
df_trafico_clean.write \
    .mode("overwrite") \
    .partitionBy("anio", "mes")\
    .parquet(ruta_trafico)

print("Datos limpios guardados correctamente")

Datos limpios guardados correctamente


In [32]:
# Verificacion final
df_silver = spark.read.parquet("gs://tfm_unir1/processed/trafico/espiras/")

df_silver.printSchema()

df_silver.agg(
    F.count("*").alias("registros"),
    F.countDistinct("id_espira").alias("espiras"),
    F.min("fecha_inicio").alias("primera_hora"),
    F.max("fecha_inicio").alias("ultima_hora"),
    F.min("intensidad").alias("minimo"),
    F.max("intensidad").alias("maximo")
    
).show(truncate=False)


root
 |-- id_espira: string (nullable = true)
 |-- fecha_inicio: timestamp (nullable = true)
 |-- intensidad: double (nullable = true)
 |-- dia: integer (nullable = true)
 |-- hora: integer (nullable = true)
 |-- dia_semana: integer (nullable = true)
 |-- anio: integer (nullable = true)
 |-- mes: integer (nullable = true)



+---------+-------+-------------------+-------------------+------+-------+
|registros|espiras|primera_hora       |ultima_hora        |minimo|maximo |
+---------+-------+-------------------+-------------------+------+-------+
|11650206 |965    |2023-03-09 16:00:00|2025-09-23 23:00:00|0.0   |12000.0|
+---------+-------+-------------------+-------------------+------+-------+

